In [1]:
!pip install pyspark
from google.colab import drive
from pyspark.sql import SparkSession

In [2]:
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
memory_limit = "12g"
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Hackathon_Silver-Gold") \
    .config("spark.driver.memory", memory_limit) \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.executor.memory", memory_limit) \
    .config("spark.memory.fraction", "0.8") \
    .getOrCreate()

In [4]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, date

from pyspark.sql.functions import col, count, when, isnan, countDistinct, approx_count_distinct, concat, col, lit, substring
from pyspark.sql.types import DoubleType, StringType, NumericType
from pyspark.sql.types import *
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

In [5]:
# --------------------------
# Função para retornar quantidade e porcentagem NULLs e NaNs
# --------------------------
def ver_nulos(dataframe):
  """
  Retorna um DataFrame Pandas ordenado com a contagem e porcentagem d
  e nulos e NaNs Performance: O(1) Action (apenas um scan na tabela).
  """
  pd.set_option('display.max_rows', 100)
  expressoes = []

  for nome_coluna, tipo_coluna in dataframe.dtypes:
      # Verifica se é float/double para checar também NaN (Not a Number)
      if tipo_coluna in ['double', 'float']:
          condicao = (F.col(nome_coluna).isNull() | isnan(F.col(nome_coluna)))
      else:
          condicao = F.col(nome_coluna).isNull()

      expressoes.append(F.count(F.when(condicao, nome_coluna)).alias(nome_coluna))

  # 2. Executa a action do Spark e converte para Pandas
  resultado = dataframe.select(expressoes).toPandas()

  # 3. Transpõe o resultado para formato de tabela (Colunas viram índices)
  resultado_final = resultado.T.rename(columns={0: 'qtd_nulos'})

  # 4. Calcula a porcentagem diretamente no Pandas
  resultado_final['pct_nulos %'] = round((resultado_final['qtd_nulos'] / dataframe.count()) * 100, 2)

  return resultado_final.sort_values('qtd_nulos', ascending=False)

In [6]:
# --------------------------
# Função para retornar o shape
# --------------------------

def get_shape(dataframe):
    linhas = f'Quanitade de linhas: {dataframe.count()}'
    colunas = f'Quanitade de Colunas: {len(dataframe.columns)}'
    return linhas, colunas

In [7]:
# --------------------------
# Função para mostrar valores agrupados da variaver
# --------------------------

def agrupamento(df, colunas):
    for col in colunas:
        df.groupBy(col).count().show(truncate = False)
        print('\n')

In [8]:
# --------------------------
# Função para retornar tabela agrupada e percentual
# --------------------------
def Freq(pTabela,pColuna):

    qtd_total=pTabela.count()

    pTabela.registerTempTable("tab_input")
    frq = spark.sql(
            """
                select
                    {col},
                    count(*) as qtd_absoluto,
                    round(100*(count(*) / {tot}),2) as qtd_percentual
                from
                    tab_input
                group by
                    {col}
                order by
                    2 desc
            """.format(col=pColuna, tot=qtd_total))

    qtd=frq.count()
    print('Quantidade de dominios',qtd)
    if qtd > 500:
        frq.show(1000,truncate=False)
        return "Dominio muito granular"

    else:
        frq.show(qtd,truncate=False)
        print("volumetria total:",qtd_total)
        return 'Freq da coluna ' + pColuna;

In [9]:
# Pastas no drive
pastas_alvo = [
    'tabela_bi_bi_dim_status_plataforma',
    'tabela_bi_dim_canal_aquisicao_credito',
    'tabela_bi_dim_forma_pagamento',
    'tabela_bi_dim_instituicao',
    'tabela_bi_dim_plano_preco',
    'tabela_bi_dim_plataforma',
    'tabela_bi_dim_promocao_credito',
    'tabela_bi_dim_tecnologia',
    'tabela_bi_dim_tipo_credito',
    'tabela_bi_dim_tipo_insercao',
    'tabela_bi_dim_tipo_recarga',
    'tabela_cadastral',
    'tabela_pagamento',
    'tabela_recarga',
    'tabela_score_bureau_full',
    'tabela_telco'
]

In [10]:
# Endereço no Drive e Endereço da pasta local
base_drive = "/content/gdrive/MyDrive/Hackathon_POD/Silver/Silver"
base_local = "/content/dados_locais"

In [11]:
# --------------------------
# Carregando os dados para a memoria local do colab
# --------------------------

for pasta in pastas_alvo:
  origem = os.path.join(base_drive, pasta)
  destino = os.path.join(base_local, pasta)

  if os.path.exists(origem):
    print(f'Procecando pasta {pasta}..')

    os.makedirs(destino, exist_ok = True)

    !cp -rn '{origem}/.' '{destino}'
    print(f'Sucesso, Copiado para: {destino}')
  else:
    print(f'ERRO: Pasta não encontrada no Drive: {origem}')

print('\n--PROCESSO FINALIZADO--')
print(f'Pastas criadas localmente: {os.listdir(base_local)}')

Procecando pasta tabela_bi_bi_dim_status_plataforma..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_bi_dim_status_plataforma
Procecando pasta tabela_bi_dim_canal_aquisicao_credito..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_canal_aquisicao_credito
Procecando pasta tabela_bi_dim_forma_pagamento..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_forma_pagamento
Procecando pasta tabela_bi_dim_instituicao..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_instituicao
Procecando pasta tabela_bi_dim_plano_preco..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_plano_preco
Procecando pasta tabela_bi_dim_plataforma..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_plataforma
Procecando pasta tabela_bi_dim_promocao_credito..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_promocao_credito
Procecando pasta tabela_bi_dim_tecnologia..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_tecnologia
Procecando pasta

In [12]:
# --------------------------
# Carregando os dados para um unico dicionario
# --------------------------
dfs = {}
path_base = "/content/dados_locais"

print('--- Carregando Bases Parquet ---')

for i in pastas_alvo:

  print(f'Carregado bases Parquet:{i}')
  dfs[i] = spark.read.parquet(f'{path_base}/{i}/*.parquet')

print(dfs.keys())

--- Carregando Bases Parquet ---
Carregado bases Parquet:tabela_bi_bi_dim_status_plataforma
Carregado bases Parquet:tabela_bi_dim_canal_aquisicao_credito
Carregado bases Parquet:tabela_bi_dim_forma_pagamento
Carregado bases Parquet:tabela_bi_dim_instituicao
Carregado bases Parquet:tabela_bi_dim_plano_preco
Carregado bases Parquet:tabela_bi_dim_plataforma
Carregado bases Parquet:tabela_bi_dim_promocao_credito
Carregado bases Parquet:tabela_bi_dim_tecnologia
Carregado bases Parquet:tabela_bi_dim_tipo_credito
Carregado bases Parquet:tabela_bi_dim_tipo_insercao
Carregado bases Parquet:tabela_bi_dim_tipo_recarga
Carregado bases Parquet:tabela_cadastral
Carregado bases Parquet:tabela_pagamento
Carregado bases Parquet:tabela_recarga
Carregado bases Parquet:tabela_score_bureau_full
Carregado bases Parquet:tabela_telco
dict_keys(['tabela_bi_bi_dim_status_plataforma', 'tabela_bi_dim_canal_aquisicao_credito', 'tabela_bi_dim_forma_pagamento', 'tabela_bi_dim_instituicao', 'tabela_bi_dim_plano_preco

In [19]:
cadastro = dfs['tabela_cadastral']
bureau = dfs['tabela_score_bureau_full']
pagamento = dfs['tabela_pagamento']
recarga = dfs['tabela_recarga']

In [ ]:
tabela_analise = spark.read.parquet('/content/gdrive/MyDrive/Hackathon_POD/AnaliseDados/tabela01.parquet/*.parquet')

# Cadastro

In [20]:
cadastro.show()

+-----------------+--------------+-----------+------+----+---------------+------------+----+--------+------------+----------------+------------------------+---------+---------------+-----------------+-------------------------+------------+------------------+--------------------------+-------------+-------------------+---------------------------+----------------+-----------------------+-------------------------------+----------+-----------------+-------------------------+--------------------+-------------------------+---------------------------------+------------+-------------------+-----------------+-------------------------+--------------------+-------------+------------------+------------------------+--------+--------------+------+--------------+------+--------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2| FPD|STATUSRF|NUM_STATUSRF|DATADENASCIMENTO|DATADENASCIMENTO_MISSING|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|SALARIO_FUNC_PUBL_M

In [22]:
df = cadastro.drop('GRUPO_CONTROLE', 'DATADENASCIMENTO_MISSING', 'SALARIO_FUNC_PUBL_MISSING', 'VALOR_EMPR_DIRETOR_MISSING', 'VALOR_EMPR_DIRETOR_MISSING',
                   'SAFRA_BOLSA_FAMILIA','SAFRA_BOLSA_FAMILIA_MISSING','BENEFICIO_BOLSA_FAMILIA_MISSING', 'NUMERO_APOSENTADO_MISSING', 'MESES_AUXILIO_EMERGENCIAL_MISSING',
                   'DATA_FUNC_PRIVADO_MISSING', 'var_07_MONETARIO', 'var_07_MONETARIO_MISSING', 'var_02','var_02_missing', 'var_03', 'var_03_MISSING',  'var_05','var_05_MISSING')

In [23]:
df.show()

+-----------------+-----------+------+----+---------------+------------+----+--------+------------+----------------+---------+---------------+-----------------+------------+------------------+-------------+----------------+-----------------------+----------+-----------------+--------------------+-------------------------+------------+-------------------+-----------------+--------------------+-------------+
|         ID_UNICO|    NUM_CPF| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2| FPD|STATUSRF|NUM_STATUSRF|DATADENASCIMENTO|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|EMPR_DIRETOR|VALOR_EMPR_DIRETOR|BOLSA_FAMILIA|UF_BOLSA_FAMILIA|BENEFICIO_BOLSA_FAMILIA|APOSENTADO|NUMERO_APOSENTADO|AUXILLIO_EMERGENCIAL|MESES_AUXILIO_EMERGENCIAL|FUNC_PRIVADO|STATUS_FUNC_PRIVADO|DATA_FUNC_PRIVADO|         CONSOLIDADO|CEP_3_digitos|
+-----------------+-----------+------+----+---------------+------------+----+--------+------------+----------------+---------+---------------+-----------------+------------+-------

In [ ]:
Freq(df, 'NUM_STATUSRF')

/usr/local/lib/python3.12/dist-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


Quantidade de dominios 7
+------------+------------+--------------+
|NUM_STATUSRF|qtd_absoluto|qtd_percentual|
+------------+------------+--------------+
|0           |3468813     |88.94         |
|1           |247171      |6.34          |
|2           |99910       |2.56          |
|3           |39738       |1.02          |
|4           |16063       |0.41          |
|-1          |15154       |0.39          |
|5           |13529       |0.35          |
+------------+------------+--------------+

volumetria total: 3900378


'Freq da coluna NUM_STATUSRF'

In [24]:
#----------------------------------------------------
# CALCULANDO IDADE DA DATA DE NASCIMENTO ATÉ A SAFRA
# ----------------------------------------------------
coluna_data_safra = F.to_date(F.concat(F.col('SAFRA'), F.lit('01')), 'yyyyMMdd')

df = df.withColumn(
    'IDADE',
    F.floor(
        F.months_between(coluna_data_safra, F.col('DATADENASCIMENTO')) / 12
    ).cast('int')
)

df = df.withColumn(
    'IDADE', F.when(F.col('DATADENASCIMENTO') == '1000-01-01', -4).otherwise(F.col('IDADE'))
).drop('DATADENASCIMENTO')

In [ ]:
df.show()

+-----------------+-----------+------+----+---------------+------------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+----------------+-----------------------+----------+-----------------+--------------------+-------------------------+------------+-------------------+-----------------+--------------------+-------------+-----+
|         ID_UNICO|    NUM_CPF| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2| FPD|STATUSRF|NUM_STATUSRF|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|EMPR_DIRETOR|VALOR_EMPR_DIRETOR|BOLSA_FAMILIA|UF_BOLSA_FAMILIA|BENEFICIO_BOLSA_FAMILIA|APOSENTADO|NUMERO_APOSENTADO|AUXILLIO_EMERGENCIAL|MESES_AUXILIO_EMERGENCIAL|FUNC_PRIVADO|STATUS_FUNC_PRIVADO|DATA_FUNC_PRIVADO|         CONSOLIDADO|CEP_3_digitos|IDADE|
+-----------------+-----------+------+----+---------------+------------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+-------------

In [ ]:
df.select('NUMERO_APOSENTADO').describe().show()

+-------+------------------+
|summary| NUMERO_APOSENTADO|
+-------+------------------+
|  count|           3900378|
|   mean|-799.8577945522203|
| stddev|410.63586359444787|
|    min|              -999|
|    max|                98|
+-------+------------------+



In [ ]:
Freq(df, 'CEP_3_digitos')

Quantidade de dominios 923
+-------------+------------+--------------+
|CEP_3_digitos|qtd_absoluto|qtd_percentual|
+-------------+------------+--------------+
|XXX          |292051      |7.49          |
|690          |64987       |1.67          |
|130          |31737       |0.81          |
|291          |30344       |0.78          |
|768          |28723       |0.74          |
|650          |28152       |0.72          |
|134          |26167       |0.67          |
|790          |25500       |0.65          |
|230          |24723       |0.63          |
|227          |24184       |0.62          |
|780          |22071       |0.57          |
|640          |21075       |0.54          |
|570          |21031       |0.54          |
|132          |20598       |0.53          |
|140          |20592       |0.53          |
|131          |20063       |0.51          |
|660          |19019       |0.49          |
|699          |18623       |0.48          |
|071          |18397       |0.47          |
|244 

'Dominio muito granular'

In [25]:
df = df.withColumn('VALOR_EMPR_DIRETOR', F.when(F.col('VALOR_EMPR_DIRETOR') == -999, -1).otherwise(F.col('VALOR_EMPR_DIRETOR')))

In [26]:
df = df.withColumn('NUMERO_APOSENTADO', F.when(F.col('NUMERO_APOSENTADO') == -999, -1).otherwise(F.col('NUMERO_APOSENTADO')))

In [ ]:
df.show()

+-----------------+-----------+------+----+---------------+------------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+----------------+-----------------------+----------+-----------------+--------------------+-------------------------+------------+-------------------+-----------------+--------------------+-------------+-----+
|         ID_UNICO|    NUM_CPF| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2| FPD|STATUSRF|NUM_STATUSRF|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|EMPR_DIRETOR|VALOR_EMPR_DIRETOR|BOLSA_FAMILIA|UF_BOLSA_FAMILIA|BENEFICIO_BOLSA_FAMILIA|APOSENTADO|NUMERO_APOSENTADO|AUXILLIO_EMERGENCIAL|MESES_AUXILIO_EMERGENCIAL|FUNC_PRIVADO|STATUS_FUNC_PRIVADO|DATA_FUNC_PRIVADO|         CONSOLIDADO|CEP_3_digitos|IDADE|
+-----------------+-----------+------+----+---------------+------------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+-------------

In [ ]:
df.columns

['ID_UNICO',
 'NUM_CPF',
 'SAFRA',
 'PROD',
 'FLAG_INSTALACAO',
 'flag_mig2',
 'FPD',
 'STATUSRF',
 'NUM_STATUSRF',
 'FUNC_PUBL',
 'CARGO_FUNC_PUBL',
 'SALARIO_FUNC_PUBL',
 'EMPR_DIRETOR',
 'VALOR_EMPR_DIRETOR',
 'BOLSA_FAMILIA',
 'UF_BOLSA_FAMILIA',
 'BENEFICIO_BOLSA_FAMILIA',
 'APOSENTADO',
 'NUMERO_APOSENTADO',
 'AUXILLIO_EMERGENCIAL',
 'MESES_AUXILIO_EMERGENCIAL',
 'FUNC_PRIVADO',
 'STATUS_FUNC_PRIVADO',
 'DATA_FUNC_PRIVADO',
 'CONSOLIDADO',
 'CEP_3_digitos',
 'IDADE']

In [27]:
ordem = ['ID_UNICO',
 'NUM_CPF',
 'IDADE',
 'SAFRA',
 'PROD',
 'FLAG_INSTALACAO',
 'flag_mig2',
 'FPD',
 'STATUSRF',
 'NUM_STATUSRF',
 'FUNC_PUBL',
 'CARGO_FUNC_PUBL',
 'SALARIO_FUNC_PUBL',
 'EMPR_DIRETOR',
 'VALOR_EMPR_DIRETOR',
 'BOLSA_FAMILIA',
 'UF_BOLSA_FAMILIA',
 'BENEFICIO_BOLSA_FAMILIA',
 'APOSENTADO',
 'NUMERO_APOSENTADO',
 'AUXILLIO_EMERGENCIAL',
 'MESES_AUXILIO_EMERGENCIAL',
 'FUNC_PRIVADO',
 'STATUS_FUNC_PRIVADO',
 'DATA_FUNC_PRIVADO',
 'CONSOLIDADO',
 'CEP_3_digitos'
 ]

In [28]:
cadastro = df[ordem]

In [29]:
cadastro.show()

+-----------------+-----------+-----+------+----+---------------+------------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+----------------+-----------------------+----------+-----------------+--------------------+-------------------------+------------+-------------------+-----------------+--------------------+-------------+
|         ID_UNICO|    NUM_CPF|IDADE| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2| FPD|STATUSRF|NUM_STATUSRF|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|EMPR_DIRETOR|VALOR_EMPR_DIRETOR|BOLSA_FAMILIA|UF_BOLSA_FAMILIA|BENEFICIO_BOLSA_FAMILIA|APOSENTADO|NUMERO_APOSENTADO|AUXILLIO_EMERGENCIAL|MESES_AUXILIO_EMERGENCIAL|FUNC_PRIVADO|STATUS_FUNC_PRIVADO|DATA_FUNC_PRIVADO|         CONSOLIDADO|CEP_3_digitos|
+-----------------+-----------+-----+------+----+---------------+------------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+-------

# Bureau

In [30]:
bureau.show()

+-----------------+--------------+-----------+------+---------------+----+----+------------+--------+----------------+--------+----------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|FLAG_INSTALACAO|PROD| FPD|   flag_mig2|SCORE_01|SCORE_01_MISSING|SCORE_02|SCORE_02_MISSING|
+-----------------+--------------+-----------+------+---------------+----+----+------------+--------+----------------+--------+----------------+
|ZZZZZZZX7T9202410|          true|ZZZZZZZX7T9|202410|              1| CMV|   1|   Aquisição|       2|               0|       1|               0|
|ZZZZZZZ8TZ8202410|          true|ZZZZZZZ8TZ8|202410|              0| CMV|NULL|SEM_MIGRACAO|     562|               0|     559|               0|
|ZZZZZZW9XWN202410|         false|ZZZZZZW9XWN|202410|              0| CMV|NULL|SEM_MIGRACAO|     585|               0|     559|               0|
|ZZZZZX7XWY8202410|         false|ZZZZZX7XWY8|202410|              1| CMV|   0|         PRE|     562|               0|     636|   

In [31]:
bureau.select('SCORE_01', 'SCORE_02').describe().show()

+-------+-----------------+------------------+
|summary|         SCORE_01|          SCORE_02|
+-------+-----------------+------------------+
|  count|          3795310|           3795310|
|   mean|570.7350572153526| 628.2029352016041|
| stddev|96.64846297203722|113.39436128638393|
|    min|               -1|                -1|
|    max|              778|               926|
+-------+-----------------+------------------+



In [ ]:
get_shape(bureau)

('Quanitade de linhas: 3795310', 'Quanitade de Colunas: 12')

In [ ]:
get_shape(cadastro)

('Quanitade de linhas: 3900378', 'Quanitade de Colunas: 27')

#Tabela

In [32]:
tabela_analise = cadastro.join(
    bureau.select('ID_UNICO','SCORE_01', 'SCORE_02'),
    on = 'ID_UNICO',
    how = 'left'
)

tabela_analise = tabela_analise.fillna(-1, subset=['SCORE_01', 'SCORE_02'])

In [ ]:
tabela_analise.show()

+-----------------+-----------+-----+------+----+---------------+------------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+----------------+-----------------------+----------+-----------------+--------------------+-------------------------+------------+-------------------+-----------------+--------------------+-------------+--------+--------+
|         ID_UNICO|    NUM_CPF|IDADE| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2| FPD|STATUSRF|NUM_STATUSRF|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|EMPR_DIRETOR|VALOR_EMPR_DIRETOR|BOLSA_FAMILIA|UF_BOLSA_FAMILIA|BENEFICIO_BOLSA_FAMILIA|APOSENTADO|NUMERO_APOSENTADO|AUXILLIO_EMERGENCIAL|MESES_AUXILIO_EMERGENCIAL|FUNC_PRIVADO|STATUS_FUNC_PRIVADO|DATA_FUNC_PRIVADO|         CONSOLIDADO|CEP_3_digitos|SCORE_01|SCORE_02|
+-----------------+-----------+-----+------+----+---------------+------------+----+--------+------------+---------+---------------+-----------------+------------+----

In [ ]:
tabela_analise.columns

['ID_UNICO',
 'NUM_CPF',
 'IDADE',
 'SAFRA',
 'PROD',
 'FLAG_INSTALACAO',
 'flag_mig2',
 'FPD',
 'STATUSRF',
 'NUM_STATUSRF',
 'FUNC_PUBL',
 'CARGO_FUNC_PUBL',
 'SALARIO_FUNC_PUBL',
 'EMPR_DIRETOR',
 'VALOR_EMPR_DIRETOR',
 'BOLSA_FAMILIA',
 'UF_BOLSA_FAMILIA',
 'BENEFICIO_BOLSA_FAMILIA',
 'APOSENTADO',
 'NUMERO_APOSENTADO',
 'AUXILLIO_EMERGENCIAL',
 'MESES_AUXILIO_EMERGENCIAL',
 'FUNC_PRIVADO',
 'STATUS_FUNC_PRIVADO',
 'DATA_FUNC_PRIVADO',
 'CONSOLIDADO',
 'CEP_3_digitos',
 'SCORE_01',
 'SCORE_02']

In [33]:
ordem2 = ['ID_UNICO',
 'NUM_CPF',
 'IDADE',
 'SAFRA',
 'PROD',
 'FLAG_INSTALACAO',
 'flag_mig2',
 'SCORE_01',
 'SCORE_02',
 'FPD',
 'STATUSRF',
 'NUM_STATUSRF',
 'FUNC_PUBL',
 'CARGO_FUNC_PUBL',
 'SALARIO_FUNC_PUBL',
 'EMPR_DIRETOR',
 'VALOR_EMPR_DIRETOR',
 'BOLSA_FAMILIA',
 'UF_BOLSA_FAMILIA',
 'BENEFICIO_BOLSA_FAMILIA',
 'APOSENTADO',
 'NUMERO_APOSENTADO',
 'AUXILLIO_EMERGENCIAL',
 'MESES_AUXILIO_EMERGENCIAL',
 'FUNC_PRIVADO',
 'STATUS_FUNC_PRIVADO',
 'DATA_FUNC_PRIVADO',
 'CONSOLIDADO',
 'CEP_3_digitos'
]

In [34]:
tabela_analise = tabela_analise[ordem2]

In [35]:
tabela_analise.show()

+-----------------+-----------+-----+------+----+---------------+------------+--------+--------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+----------------+-----------------------+----------+-----------------+--------------------+-------------------------+------------+-------------------+-----------------+--------------------+-------------+
|         ID_UNICO|    NUM_CPF|IDADE| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2|SCORE_01|SCORE_02| FPD|STATUSRF|NUM_STATUSRF|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|EMPR_DIRETOR|VALOR_EMPR_DIRETOR|BOLSA_FAMILIA|UF_BOLSA_FAMILIA|BENEFICIO_BOLSA_FAMILIA|APOSENTADO|NUMERO_APOSENTADO|AUXILLIO_EMERGENCIAL|MESES_AUXILIO_EMERGENCIAL|FUNC_PRIVADO|STATUS_FUNC_PRIVADO|DATA_FUNC_PRIVADO|         CONSOLIDADO|CEP_3_digitos|
+-----------------+-----------+-----+------+----+---------------+------------+--------+--------+----+--------+------------+---------+---------------+-----------------

In [ ]:
Freq(tabela_analise, 'flag_mig2')

Quantidade de dominios 4
+------------+------------+--------------+
|flag_mig2   |qtd_absoluto|qtd_percentual|
+------------+------------+--------------+
|Aquisição   |1338888     |34.33         |
|PRE         |1290526     |33.09         |
|SEM_MIGRACAO|1266478     |32.47         |
|FLEX        |4486        |0.12          |
+------------+------------+--------------+

volumetria total: 3900378


'Freq da coluna flag_mig2'

In [ ]:
Freq(tabela_analise, 'PROD')

Quantidade de dominios 3
+----+------------+--------------+
|PROD|qtd_absoluto|qtd_percentual|
+----+------------+--------------+
|CMV |3795310     |97.31         |
|NET |89968       |2.31          |
|DTH |15100       |0.39          |
+----+------------+--------------+

volumetria total: 3900378


'Freq da coluna PROD'

In [ ]:
path_analise = "/content/gdrive/MyDrive/Hackathon_POD/AnaliseDados/tabela01.parquet"
tabela_analise.write.mode("overwrite").parquet(path_analise)

In [ ]:
tabelas_dim = [
    'tabela_bi_bi_dim_status_plataforma',
    'tabela_bi_dim_canal_aquisicao_credito',
    'tabela_bi_dim_forma_pagamento',
    'tabela_bi_dim_instituicao',
    'tabela_bi_dim_plano_preco',
    'tabela_bi_dim_plataforma',
    'tabela_bi_dim_promocao_credito',
    'tabela_bi_dim_tecnologia',
    'tabela_bi_dim_tipo_credito',
    'tabela_bi_dim_tipo_insercao',
    'tabela_bi_dim_tipo_recarga']

In [ ]:
for i in tabelas_dim:
  print(i)
  dfs[i].show(40)
  print('\n')

tabela_bi_bi_dim_status_plataforma
+---------------------+---------------------+---------+------------------+--------------+-------------------+----------------------+
|COD_STATUS_PLATAFORMA|DSC_STATUS_PLATAFORMA|IND_ATIVO|DAT_ATUALIZACAO_DW|DAT_CRIACAO_DW|COD_STATUS_PLAT_GRP|IND_STS_PLAT_GRP_ATIVO|
+---------------------+---------------------+---------+------------------+--------------+-------------------+----------------------+
|                    A|                Ativo|        S|        2006-10-16|    2006-10-16|                  A|                     S|
|                  ZB1|           Expirado 1|        S|        2006-10-16|    2006-10-16|                ZB1|                     S|
|                  PRE|            Pré-Ativo|        N|        2006-10-16|    2006-10-16|                PRE|                     N|
|                  ZB2|           Expirado 2|        N|        2006-10-16|    2006-10-16|                ZB2|                     N|
|                    C|         De

In [36]:
recarga.show()

+-----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+--------------+
|    NUM_CPF|DW_NUM_NTC|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|DW_NUM_CLIENTE|COD_TECNOLOGIA_DW|COD_CANAL_AQUISICAO|COD_TIPO_CREDITO|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAl|COD_PLATAFORMA_ATU|COD_STATUS_PLATAFORMA|IND_METODO_PAGAMENTO|DW_PLANO_TARIFACAO|DW_TIPO_RECARGA|DW_TIPO_INSERCAO|DW_FORMA_PAGAMENTO|DW_INSTITUICAO|COD_GRUPO_CARTAO|DSC_GRUPO_CARTAO_WPP|FLAG_SOS|VALOR_SOS|GRUPO_CONTROLE|
+-----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+

In [49]:
pagamento.show()

+-----------+------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+--------------------

In [43]:
from pyspark.sql import functions as F

# Supondo que seu dataframe de pagamentos se chame 'df_pagamentos'

# 1. Agrupamos por CPF e contamos quantos contratos DISTINTOS cada um tem
df_validacao_contratos = pagamento.groupBy("NUM_CPF") \
    .agg(F.countDistinct("CONTRATO").alias("qtd_contratos")) \
    .filter(F.col("qtd_contratos") > 1) # Filtramos só quem tem mais de 1

# 2. Mostramos o resultado
print(f"Quantidade de clientes com mais de 1 contrato: {df_validacao_contratos.count()}")
df_validacao_contratos.show(5)

# 3. (Opcional) Vamos ver um exemplo real para entender o comportamento
# Pegamos um CPF da lista acima (copie um CPF do show() anterior)
exemplo_cpf = "COLOQUE_UM_CPF_AQUI"

pagamento.filter(F.col("NUM_CPF") == exemplo_cpf) \
    .select("NUM_CPF", "CONTRATO", "DAT_VENCIMENTO_CREDITO", "VAL_PAGAMENTO_FATURA") \
    .orderBy("DAT_VENCIMENTO_CREDITO") \
    .show()

Quantidade de clientes com mais de 1 contrato: 582785
+-----------+-------------+
|    NUM_CPF|qtd_contratos|
+-----------+-------------+
|WNUXXWW7YN8|            2|
|ZYUYUNY7Z7Z|            2|
|UXTXWYYWXZZ|            2|
|NNNZ9Z89WTW|            3|
|Z8X8YWT87ZX|            3|
+-----------+-------------+
only showing top 5 rows
+-------+--------+----------------------+--------------------+
|NUM_CPF|CONTRATO|DAT_VENCIMENTO_CREDITO|VAL_PAGAMENTO_FATURA|
+-------+--------+----------------------+--------------------+
+-------+--------+----------------------+--------------------+



In [46]:
from pyspark.sql import functions as F

# Supondo que seu dataframe de pagamentos se chame 'df_pagamentos'

# 1. Agrupamos por CPF e contamos quantos contratos DISTINTOS cada um tem
df_validacao_contratos = pagamento.groupBy("NUM_CPF") \
    .agg(F.countDistinct("CONTRATO").alias("qtd_contratos")) \
    .filter(F.col("qtd_contratos") > 1) # Filtramos só quem tem mais de 1

# 2. Mostramos o resultado
print(f"Quantidade de clientes com mais de 1 contrato: {df_validacao_contratos.count()}")
df_validacao_contratos.show(30)

# 3. (Opcional) Vamos ver um exemplo real para entender o comportamento
# Pegamos um CPF da lista acima (copie um CPF do show() anterior)
exemplo_cpf = "U9TYWNY7ZXT"

pagamento.filter(F.col("NUM_CPF") == exemplo_cpf) \
    .select("NUM_CPF", "CONTRATO", "DAT_VENCIMENTO_CREDITO", "VAL_PAGAMENTO_FATURA") \
    .orderBy("DAT_VENCIMENTO_CREDITO") \
    .show()

Quantidade de clientes com mais de 1 contrato: 582785
+-----------+-------------+
|    NUM_CPF|qtd_contratos|
+-----------+-------------+
|WNUXXWW7YN8|            2|
|ZYUYUNY7Z7Z|            2|
|UXTXWYYWXZZ|            2|
|NNNZ9Z89WTW|            3|
|Z8X8YWT87ZX|            3|
|ZT9NYN9UXYZ|            2|
|ZNZWXY7WNWY|            2|
|ZXWY788ZXYX|            3|
|Z7XT8T7878Z|            4|
|XTXWTZUW7XW|            2|
|ZYZW8UWW8ZX|            3|
|XY8XWNU8WTW|            2|
|W8T9N9NT8Z7|            3|
|ZXXN77Z9888|            3|
|99YYZY7ZZWZ|            2|
|YXUT9TUXXYZ|            2|
|ZNNU8UYWU8Z|            3|
|ZXNW8ZT78UY|            3|
|ZXNU7NWZZ7Z|            2|
|ZUZWUZ8NXZN|            2|
|ZZ7TXUWTUYY|            3|
|ZYXZNW9ZXNT|            3|
|UNXZNT7988X|            2|
|U9TYWNY7ZXT|            4|
|ZUYX878Y8YY|            2|
|ZWYZX9T7XNZ|            4|
|ZT7W79897U7|            2|
|ZXYU7T87XX8|            2|
|ZNUWW879UW8|            2|
|NUTUXTX7TXT|            2|
+-----------+---------

In [47]:
df_resultado = pagamento.select(F.countDistinct("CONTRATO").alias("total_unicos"))

df_resultado.show()

+------------+
|total_unicos|
+------------+
|     2672803|
+------------+



In [48]:
df_resultado2 = pagamento.select(F.countDistinct("NUM_CPF").alias("total_unicos"))

df_resultado2.show()

+------------+
|total_unicos|
+------------+
|     1930502|
+------------+



In [54]:
tabela_analise.show(5)

+-----------------+-----------+-----+------+----+---------------+------------+--------+--------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+----------------+-----------------------+----------+-----------------+--------------------+-------------------------+------------+-------------------+-----------------+--------------------+-------------+
|         ID_UNICO|    NUM_CPF|IDADE| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2|SCORE_01|SCORE_02| FPD|STATUSRF|NUM_STATUSRF|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|EMPR_DIRETOR|VALOR_EMPR_DIRETOR|BOLSA_FAMILIA|UF_BOLSA_FAMILIA|BENEFICIO_BOLSA_FAMILIA|APOSENTADO|NUMERO_APOSENTADO|AUXILLIO_EMERGENCIAL|MESES_AUXILIO_EMERGENCIAL|FUNC_PRIVADO|STATUS_FUNC_PRIVADO|DATA_FUNC_PRIVADO|         CONSOLIDADO|CEP_3_digitos|
+-----------------+-----------+-----+------+----+---------------+------------+--------+--------+----+--------+------------+---------+---------------+-----------------

In [51]:
pagamento  = pagamento.withColumns({
    'DAT_STATUS_FATURA': F.to_date(F.col('DAT_STATUS_FATURA'), "ddMMMyyyy:HH:mm:ss"),
    'DAT_CRIACAO_DW' : F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss"),

})

In [52]:
pagamento.show()

+-----------+-----------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+--------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+---------------------+---

In [53]:
pagamento['NUM_CPF', 'DAT_STATUS_FATURA'].show()

+-----------+-----------------+
|    NUM_CPF|DAT_STATUS_FATURA|
+-----------+-----------------+
|ZXYTW77Z88W|       2024-10-24|
|9NWYW7Y8N7Y|       2023-10-21|
|ZUUTNYYWX89|       2024-09-07|
|ZTNW8TU8TTN|       2024-10-04|
|7NN78U8Y7ZZ|       2023-12-21|
|XYYT7ZNU7TT|       2024-11-02|
|ZWYNNZZ79TN|       2024-06-07|
|ZY8XUX98UXN|       2025-01-03|
|XYUX9UZX8T9|       2024-10-29|
|U7WYYW8X9ZU|       2025-03-21|
|XWX9XN7888W|       2024-01-10|
|ZTW9XX8T8WX|       2023-12-11|
|YWYTW7TUXYZ|       2025-02-03|
|UXY7X79X8ZW|       2025-01-17|
|Z7U7ZT97TNT|       2023-11-07|
|X9U9787U8NW|       2025-01-21|
|XYUX9UZX8T9|       2025-03-08|
|YTZXXTYW87Z|       2024-04-16|
|WYT9UXYY8Z7|       2025-02-04|
|ZTY87UXX87X|       2025-03-07|
+-----------+-----------------+
only showing top 20 rows


In [55]:
# 1. Tratamento da Tabela Pagamentos (Encontrar o inicio do relacionamento)
# Agrupamos por CPF para pegar a primeira vez que ele teve uma fatura gerada
df_primeira_fatura = pagamento \
    .filter(F.col("DAT_STATUS_FATURA") > "1900-01-01") \
    .groupBy("NUM_CPF") \
    .agg(F.min("DAT_STATUS_FATURA").alias("DATA_INICIO_POS"))

# 2. Tratamento da Tabela Análise (Preparar a data da Safra)
# Convertemos 202411 (Inteiro) para 2024-11-01 (Data) para poder comparar
df_analise_prep = tabela_analise.withColumn(
    "DATA_REF_SAFRA",
    F.to_date(F.col("SAFRA").cast("string"), "yyyyMM")
)

# 3. Join entre Análise e Histórico de Pagamentos
df_enrich = df_analise_prep.join(df_primeira_fatura, on="NUM_CPF", how="left")

# 4. Criação da Flag POSSUI_POS (Anti-Leakage)
df_final = df_enrich.withColumn("POSSUI_POS",
    F.when(
        # Se existe uma data de inicio E ela é ANTERIOR à data da safra atual...
        (F.col("DATA_INICIO_POS").isNotNull()) &
        (F.col("DATA_INICIO_POS") < F.col("DATA_REF_SAFRA")),
        1 # ...significa que ele JÁ ERA cliente Pós antes dessa safra.
    ).otherwise(0)
).drop("DATA_REF_SAFRA", "DATA_INICIO_POS") # Removemos as colunas auxiliares

# Visualização do Resultado
df_final.select("ID_UNICO", "NUM_CPF", "SAFRA", "POSSUI_POS").show(10)

+-----------------+-----------+------+----------+
|         ID_UNICO|    NUM_CPF| SAFRA|POSSUI_POS|
+-----------------+-----------+------+----------+
|77789989YZZ202503|77789989YZZ|202503|         1|
|7778TYWXT9X202412|7778TYWXT9X|202412|         0|
|7778XUZNYU9202410|7778XUZNYU9|202410|         1|
|7778YY8WZZZ202411|7778YY8WZZZ|202411|         0|
|7779TX8ZTXT202410|7779TX8ZTXT|202410|         0|
|7779XWW98YZ202502|7779XWW98YZ|202502|         0|
|7779XWW98YZ202503|7779XWW98YZ|202503|         0|
|7779ZY78YXT202411|7779ZY78YXT|202411|         0|
|777N7NYU8YZ202411|777N7NYU8YZ|202411|         1|
|777NTUNZZWZ202503|777NTUNZZWZ|202503|         0|
+-----------------+-----------+------+----------+
only showing top 10 rows


In [56]:
df_final.show()

+-----------+-----------------+-----+------+----+---------------+------------+--------+--------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+----------------+-----------------------+----------+-----------------+--------------------+-------------------------+------------+-------------------+-----------------+--------------------+-------------+----------+
|    NUM_CPF|         ID_UNICO|IDADE| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2|SCORE_01|SCORE_02| FPD|STATUSRF|NUM_STATUSRF|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|EMPR_DIRETOR|VALOR_EMPR_DIRETOR|BOLSA_FAMILIA|UF_BOLSA_FAMILIA|BENEFICIO_BOLSA_FAMILIA|APOSENTADO|NUMERO_APOSENTADO|AUXILLIO_EMERGENCIAL|MESES_AUXILIO_EMERGENCIAL|FUNC_PRIVADO|STATUS_FUNC_PRIVADO|DATA_FUNC_PRIVADO|         CONSOLIDADO|CEP_3_digitos|POSSUI_POS|
+-----------+-----------------+-----+------+----+---------------+------------+--------+--------+----+--------+------------+---------+-----------

In [58]:
path_analise = "/content/gdrive/MyDrive/Hackathon_POD/AnaliseDados/tabela_analise.parquet"
df_final.write.mode("overwrite").parquet(path_analise)

In [ ]:
df_analise = tabela_analise
df_recarga = recarga
df_dim_plataforma = dfs['tabela_bi_dim_plataforma']

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- 1. Preparação das Tabelas ---
# Assumindo que você já tem df_analise, df_recarga e df_dim_plataforma carregados

# Converter SAFRA para Data (Dia 01 do mês) para fazer a comparação
df_analise_temp = df_analise.withColumn(
    'DATA_SAFRA',
    F.to_date(F.concat(F.col('SAFRA'), F.lit('01')), 'yyyyMMdd')
).select('ID_UNICO', 'NUM_CPF', 'DATA_SAFRA') # Só precisamos das chaves agora

# --- 2. Busca da Última Plataforma (Snapshot) ---
# Join com Recarga
df_join = df_analise_temp.join(df_recarga, on='NUM_CPF', how='inner')

# Filtro Anti-Leakage: Só recargas ANTES da Safra
df_validas = df_join.filter(F.col('DAT_INSERCAO_CREDITO') < F.col('DATA_SAFRA'))

# Window Function para pegar a MAIS RECENTE
w = Window.partitionBy('ID_UNICO').orderBy(F.col('DAT_INSERCAO_CREDITO').desc())

df_ultima_plat = df_validas.withColumn('rn', F.row_number().over(w)) \
    .filter(F.col('rn') == 1) \
    .select('ID_UNICO', F.col('COD_PLATAFORMA_ATU').alias('COD_PLAT_RECENTE'))

# --- 3. Enriquecimento com a Dimensão ---
# Preparar a dimensão (Selecionar colunas e renomear para facilitar o join)
df_dim = df_dim_plataforma.select(
    F.col('DSC_PLATAFORMA').alias('COD_PLAT_RECENTE'),
    F.col('DSC_GRUPO_PLATAFORMA')
)

# Join com a Dimensão
df_plat_enrich = df_ultima_plat.join(df_dim, on='COD_PLAT_RECENTE', how='left')

# --- 4. Categorização Final ---
df_plat_final = df_plat_enrich.withColumn(
    'PLATAFORMA',
    F.when(F.col('DSC_GRUPO_PLATAFORMA') == 'Pré Pago', 'PRE_PAGO')
     .when(F.col('DSC_GRUPO_PLATAFORMA') == 'Controle', 'CONTROLE')
     .when(F.col('DSC_GRUPO_PLATAFORMA') == 'Pós Pago', 'POS_PAGO')
     .otherwise('OUTROS')
).select('ID_UNICO', 'PLATAFORMA')

# --- 5. Join Final com a Tabela de Análise ---
# Left Join para manter todos da tabela original. Quem não tem recarga vira 'DESCONHECIDO'
df_analise_atualizada = df_analise.join(df_plat_final, on='ID_UNICO', how='left') \
    .fillna('DESCONHECIDO', subset=['PLATAFORMA'])

# Exibir resultado
df_analise_atualizada.select('ID_UNICO', 'SAFRA', 'PLATAFORMA').show(5)

+-----------------+------+------------+
|         ID_UNICO| SAFRA|  PLATAFORMA|
+-----------------+------+------------+
|777777UWTYZ202502|202502|    PRE_PAGO|
|77777U9YN9X202503|202503|    PRE_PAGO|
|77778TXZNWU202410|202410|DESCONHECIDO|
|77778Z9ZUZN202503|202503|    PRE_PAGO|
|77779Z9NU9X202410|202410|DESCONHECIDO|
+-----------------+------+------------+
only showing top 5 rows


In [ ]:
df_analise_atualizada.show()

+-----------------+-----------+-----+------+----+---------------+------------+--------+--------+----+--------+------------+---------+---------------+-----------------+------------+------------------+-------------+----------------+-----------------------+----------+-----------------+--------------------+-------------------------+------------+-------------------+-----------------+--------------------+-------------+------------+
|         ID_UNICO|    NUM_CPF|IDADE| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2|SCORE_01|SCORE_02| FPD|STATUSRF|NUM_STATUSRF|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|EMPR_DIRETOR|VALOR_EMPR_DIRETOR|BOLSA_FAMILIA|UF_BOLSA_FAMILIA|BENEFICIO_BOLSA_FAMILIA|APOSENTADO|NUMERO_APOSENTADO|AUXILLIO_EMERGENCIAL|MESES_AUXILIO_EMERGENCIAL|FUNC_PRIVADO|STATUS_FUNC_PRIVADO|DATA_FUNC_PRIVADO|         CONSOLIDADO|CEP_3_digitos|  PLATAFORMA|
+-----------------+-----------+-----+------+----+---------------+------------+--------+--------+----+--------+------------+---------+-------

In [ ]:
Freq(df_analise_atualizada, 'PLATAFORMA')

Quantidade de dominios 5
+------------+------------+--------------+
|PLATAFORMA  |qtd_absoluto|qtd_percentual|
+------------+------------+--------------+
|PRE_PAGO    |1976786     |50.68         |
|DESCONHECIDO|1181916     |30.3          |
|CONTROLE    |684025      |17.54         |
|POS_PAGO    |57612       |1.48          |
|OUTROS      |39          |0.0           |
+------------+------------+--------------+

volumetria total: 3900378


'Freq da coluna PLATAFORMA'

In [ ]:
get_shape(df_analise_atualizada)

('Quanitade de linhas: 3900378', 'Quanitade de Colunas: 30')

# Fim

In [ ]:
df_analise_atualizada.select('PROD', 'FLAG_INSTALACAO', 'flag_mig2', 'PLATAFORMA').show(200)

+----+---------------+------------+------------+
|PROD|FLAG_INSTALACAO|   flag_mig2|  PLATAFORMA|
+----+---------------+------------+------------+
| CMV|              0|SEM_MIGRACAO|    PRE_PAGO|
| CMV|              1|         PRE|    PRE_PAGO|
| CMV|              1|   Aquisição|DESCONHECIDO|
| NET|              0|SEM_MIGRACAO|    PRE_PAGO|
| CMV|              0|SEM_MIGRACAO|DESCONHECIDO|
| CMV|              1|         PRE|    CONTROLE|
| CMV|              1|   Aquisição|DESCONHECIDO|
| CMV|              1|         PRE|    PRE_PAGO|
| CMV|              0|SEM_MIGRACAO|    PRE_PAGO|
| CMV|              0|SEM_MIGRACAO|    POS_PAGO|
| CMV|              1|         PRE|    PRE_PAGO|
| CMV|              1|   Aquisição|DESCONHECIDO|
| CMV|              0|SEM_MIGRACAO|DESCONHECIDO|
| CMV|              1|   Aquisição|DESCONHECIDO|
| CMV|              1|         PRE|    PRE_PAGO|
| CMV|              1|   Aquisição|DESCONHECIDO|
| CMV|              1|   Aquisição|DESCONHECIDO|
| CMV|              

In [ ]:
path_analise = "/content/gdrive/MyDrive/Hackathon_POD/AnaliseDados/tabela02.parquet"
df_analise_atualizada.write.mode("overwrite").parquet(path_analise)

In [ ]:
recarga.show()

+-----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+--------------+
|    NUM_CPF|DW_NUM_NTC|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|DW_NUM_CLIENTE|COD_TECNOLOGIA_DW|COD_CANAL_AQUISICAO|COD_TIPO_CREDITO|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAl|COD_PLATAFORMA_ATU|COD_STATUS_PLATAFORMA|IND_METODO_PAGAMENTO|DW_PLANO_TARIFACAO|DW_TIPO_RECARGA|DW_TIPO_INSERCAO|DW_FORMA_PAGAMENTO|DW_INSTITUICAO|COD_GRUPO_CARTAO|DSC_GRUPO_CARTAO_WPP|FLAG_SOS|VALOR_SOS|GRUPO_CONTROLE|
+-----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+

In [ ]:
df_recarga = recarga['HOR_INSERCAO_CREDITO', 'VAL_CREDITO_INSERIDO', 'VAL_BONUS', 'VAL_REAl', 'FLAG_SOS', 'VALOR_SOS']

In [ ]:
df_recarga.show()

+--------------------+--------------------+---------+--------+--------+---------+
|HOR_INSERCAO_CREDITO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAl|FLAG_SOS|VALOR_SOS|
+--------------------+--------------------+---------+--------+--------+---------+
|              105734|                20.0|      0.0|    20.0|       0|      0.0|
|              164634|                 0.0|      1.0|     1.0|       0|      0.0|
|              113537|                 0.0|      0.0|     0.0|       0|      0.0|
|                1105|                 0.0|      0.0|     0.0|       0|      0.0|
|              100036|                 0.0|   8200.0|  8200.0|       0|      0.0|
|               73739|                39.9|    -38.9|     1.0|       0|      0.0|
|              175244|                 0.0|  80500.0| 80500.0|       0|      0.0|
|              112454|                30.0|      0.0|    30.0|       0|      0.0|
|              233035|                 0.0|     -1.0|    -1.0|       0|      0.0|
|               

In [ ]:
path_analise = "/content/gdrive/MyDrive/Hackathon_POD/AnaliseDados/recarga.parquet"
df_recarga.write.mode("overwrite").parquet(path_analise)